In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

## Ingestion del carpeta "movie_language"

###Paso 1 - Leer los archivos JSON usando "DataframeReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

In [0]:
movies_languages_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("languageId", IntegerType(), False),
    StructField("languageRoleId", IntegerType(), False)
])

In [0]:
movies_languages_df = spark.read\
    .option("multiLine", True)\
    .schema(movies_languages_schema)\
    .json(f"{bronze_folder_path}/{v_file_date}/movie_language")

### Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import lit, current_timestamp

In [0]:
movies_languages_final_df = add_ingestion_date(movies_languages_df)\
    .withColumnsRenamed({"movieId": "movie_id",
                         "languageId": "language_id"})\
    .withColumn("environment", lit("Production"))\
    .withColumn("file_date", lit(v_file_date))\
    .drop("languageRoleId")

### Paso 3 - Escribir la salida en un formato "Parquet" PartitionBy

In [0]:
#overwrite_partition("movie_silver", "movies_languages", "file_date", v_file_date)

In [0]:
#movies_languages_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/movies_languages")

In [0]:
#movies_languages_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movies_languages")

condition_merge = 'tgt.movie_id = src.movie_id AND tgt.language_id = src.language_id AND tgt.file_date = src.file_date'

incremental_merge("movie_silver", "movies_languages", movies_languages_final_df, condition_merge, "file_date")

In [0]:
%sql
SELECT file_date, count(1)
FROM movie_silver.movies_languages
GROUP BY file_date;

file_date,count(1)
2024-12-16,6000
2024-12-23,3010
2024-12-30,2730


In [0]:
dbutils.notebook.exit("Exitoso")